# Join Amenities and School Counts to Resale Transactions
This notebook takes the cleaned resale panel and enriches it with nearby malls, rail access, and school exposure measures. The final output is the transaction-level dataset used by the DiD and RDD model pipelines.


In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
from pyproj import Transformer

PROCESSED_DIR = Path("../data/processed")

resale_df = pd.read_csv(PROCESSED_DIR / "cleaned_hdb_resale.csv")
malls_df = pd.read_csv(PROCESSED_DIR / "malls_with_coords.csv")
mrt_df = pd.read_csv(PROCESSED_DIR / "mrt_lrt_data.csv")

## Load Processed Inputs and Set Distance Rules
The setup cells load the cleaned resale panel together with the processed malls and rail datasets, then define the neighborhood radii used for the proximity counts.


In [ ]:
# define these metrics in KM 
malls_nearby_distance = 1
mrt_nearby_distance = 1

## Prepare Dates and Geometry Inputs
Before computing joins, the notebook constructs a transaction date for each resale record, parses mall close dates, and converts rail geometries into latitude-longitude coordinates for distance calculations.


In [ ]:
# Create resale date column, take month and year and set date to 1
resale_df['Date'] = pd.to_datetime(resale_df[['year', 'month']].assign(Day=1))
# convert malls_df close date to date time
malls_df['close_date'] = pd.to_datetime(malls_df['close_date'])

# convert geometry in mrt_lrt to lat lon
transformer = Transformer.from_crs("EPSG:3414", "EPSG:4326", always_xy=True)

mrt_df[["lon", "lat"]] = mrt_df["geometry"].str.extract(r"POINT \((.*) (.*)\)").astype(float).apply(
    lambda row: pd.Series(transformer.transform(row[0], row[1])),
    axis=1
)
mrt_df['Date'] = pd.to_datetime(mrt_df['Date'])

In [ ]:
mrt_df

Join data and count total number of malls/mrt near the area. if want 1 only then can just max(1,x) later on 



## Add Mall Exposure Measures
This block computes HDB-to-mall distances, applies the mall opening-status condition at the transaction date, and counts how many eligible malls are within the chosen radius.


## Join resale and malls

In [ ]:
# initialise transformer once (faster)
transformer_3414 = Transformer.from_crs("EPSG:4326", "EPSG:3414", always_xy=True)

def distance_matrix_3414(lat1, lon1, lat2, lon2):
    """
    Returns pairwise distance matrix in km
    using SVY21 (EPSG:3414) projected coordinates.
    """
    # convert to numpy arrays
    lat1 = np.array(lat1)
    lon1 = np.array(lon1)
    lat2 = np.array(lat2)
    lon2 = np.array(lon2)

    # transform to SVY21 (x = Easting, y = Northing)
    x1, y1 = transformer_3414.transform(lon1, lat1)
    x2, y2 = transformer_3414.transform(lon2, lat2)

    # reshape for broadcasting
    x1 = x1[:, None]
    y1 = y1[:, None]
    x2 = x2[None, :]
    y2 = y2[None, :]

    # Euclidean distance (in meters)
    dist_m = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    # convert to km
    return dist_m / 1000

In [ ]:

# compute HDB-to-mall distance matrix
dist_matrix = distance_matrix_3414(
    resale_df["latitude"].to_numpy(),
    resale_df["longitude"].to_numpy(),
    malls_df["lat"].to_numpy(),
    malls_df["lon"].to_numpy()
)

# --- Distance condition ---
within_distance = dist_matrix <= malls_nearby_distance  

# --- Time condition ---
# mall is valid if:
# no close_date (still open), OR closed AFTER resale_date

resale_dates = resale_df["Date"].values[:, None]   # (n_hdb x 1)
close_dates = malls_df["close_date"].values[None, :]    # (1 x n_mall)

mall_open = (np.isnat(close_dates)) | (close_dates >= resale_dates)

# --- Combine both conditions ---
valid = within_distance & mall_open

# --- Count per HDB ---
resale_df["num_nearby_malls"] = valid.sum(axis=1)

In [ ]:
resale_dates

## Add Rail Exposure Measures
This section repeats the same logic for MRT and LRT stations, combining the distance filter with station opening dates so only operating stations count toward access measures.


## Join resale and mrt

In [ ]:

# distance from each resale flat to each MRT
dist_matrix = distance_matrix_3414(
    resale_df["latitude"].to_numpy(),
    resale_df["longitude"].to_numpy(),
    mrt_df["lat"].to_numpy(),
    mrt_df["lon"].to_numpy()
)

# condition 1: within radius
within_distance = dist_matrix <= mrt_nearby_distance

# condition 2: MRT already existed by transaction date
resale_dates = resale_df["Date"].values[:, None]   # shape (n_resale, 1)
mrt_dates = mrt_df["Date"].values[None, :]         # shape (1, n_mrt)

date_valid = resale_dates >= mrt_dates

# combine both conditions
valid = within_distance & date_valid

# count nearby MRTs for each resale row
resale_df["num_nearby_mrt"] = valid.sum(axis=1)

# count unique mrt_line values for each resale row
mrt_lines = mrt_df["Line_name"].fillna("__MISSING__").to_numpy()

resale_df["num_unique_mrt_lines"] = [
    len(np.unique(mrt_lines[row_mask & (mrt_lines != "__MISSING__")]))
    for row_mask in valid
]

## Add School Exposure Measures
The school join uses the processed school boundary file to count how many schools and good schools are near each resale transaction under both polygon-based and point-based distance definitions.


## Add school data

In [ ]:
import geopandas as gpd

gdf = gpd.read_file("../data/processed/schools/final_primary_schools_with_school_boundaries.geojson")
good_school_df = pd.read_csv("../data/processed/schools/school_admissions_no_gep_sap.csv")

gdf["good_school"] = gdf["school_name"].isin(good_school_df["School"])

DIST = 1000  # 1 km in meters

# ----------------------------
# 1) resale points from lat/lon
# ----------------------------
resale_gdf = gpd.GeoDataFrame(
    resale_df.copy(),
    geometry=gpd.points_from_xy(resale_df["longitude"], resale_df["latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:3414")

# ----------------------------
# 2) prepare school data
# ----------------------------
school_gdf = gdf.copy()
school_gdf["start_year"] = pd.to_numeric(school_gdf["start_year"], errors="coerce")
school_gdf["end_year"] = pd.to_numeric(school_gdf["end_year"], errors="coerce").fillna(np.inf)

# method 1: polygon geometry
school_poly_gdf = gpd.GeoDataFrame(
    school_gdf.copy(),
    geometry="geometry",
    crs="EPSG:3414"
)

good_school_poly_gdf = gpd.GeoDataFrame(
    school_gdf[school_gdf["good_school"]].copy(),
    geometry="geometry",
    crs="EPSG:3414"
)

# method 2: point from X,Y
school_point_gdf = gpd.GeoDataFrame(
    school_gdf.copy(),
    geometry=gpd.points_from_xy(school_gdf["X"], school_gdf["Y"]),
    crs="EPSG:3414"
)

good_school_df = school_gdf[school_gdf["good_school"]].copy()

good_school_point_gdf = gpd.GeoDataFrame(
    good_school_df,
    geometry=gpd.points_from_xy(good_school_df["X"], good_school_df["Y"]),
    crs="EPSG:3414"
)


In [ ]:
# ----------------------------
# counting functions
# ----------------------------
def count_schools_polygon(row, min_dist=0, max_dist=1000, good=0):
    year = row["year"]
    pt = row.geometry

    schools = good_school_poly_gdf if good == 1 else school_poly_gdf

    active = schools[
        (schools["start_year"] <= year) &
        (year <= schools["end_year"])
    ]

    if active.empty:
        return 0

    dists = active.geometry.distance(pt)
    return int(((dists > min_dist) & (dists <= max_dist)).sum())


def count_schools_xy(row, min_dist=0, max_dist=1000, good=0):
    year = row["year"]
    pt = row.geometry

    schools = good_school_point_gdf if good == 1 else school_point_gdf

    active = schools[
        (schools["start_year"] <= year) &
        (year <= schools["end_year"])
    ]

    if active.empty:
        return 0

    dists = active.geometry.distance(pt)
    return int(((dists > min_dist) & (dists <= max_dist)).sum())

In [ ]:
# ----------------------------
# create count columns schools within 1km
# ----------------------------
resale_gdf["num_schools_0_1km_polygon"] = resale_gdf.apply(
    lambda row: count_schools_polygon(row, min_dist=0, max_dist=1000,good = 0),
    axis=1
)

resale_gdf["num_schools_0_1km_xy"] = resale_gdf.apply(
    lambda row: count_schools_xy(row, min_dist=0, max_dist=1000, good = 0),
    axis=1
)

# ----------------------------
# create count columns schools btw 1-2km
# ----------------------------
resale_gdf["num_schools_1_2km_polygon"] = resale_gdf.apply(
    lambda row: count_schools_polygon(row, min_dist=1000, max_dist=2000, good = 0),
    axis=1
)

resale_gdf["num_schools_1_2km_xy"] = resale_gdf.apply(
    lambda row: count_schools_xy(row, min_dist=1000, max_dist=2000, good = 0),
    axis=1
)

# ----------------------------
# create count columns good schools within 1km
# ----------------------------
resale_gdf["num_good_schools_0_1km_polygon"] = resale_gdf.apply(
    lambda row: count_schools_polygon(row, min_dist=0, max_dist=1000, good = 1),
    axis=1
)

resale_gdf["num_good_schools_0_1km_xy"] = resale_gdf.apply(
    lambda row: count_schools_xy(row, min_dist=0, max_dist=1000, good = 1),
    axis=1
)

# ----------------------------
# create count columns good schools btw 1-2km
# ----------------------------
resale_gdf["num_good_schools_1_2km_polygon"] = resale_gdf.apply(
    lambda row: count_schools_polygon(row, min_dist=1000, max_dist=2000, good = 1),
    axis=1
)

resale_gdf["num_good_schools_1_2km_xy"] = resale_gdf.apply(
    lambda row: count_schools_xy(row, min_dist=1000, max_dist=2000, good = 1),
    axis=1
)


## Export Final Joined Dataset
The finished transaction panel, now enriched with amenity and school counts, is written to data/processed/final_resale_data.csv.


In [ ]:
resale_gdf.to_csv("../data/processed/final_resale_data.csv", index=False)